# Notebook 3 · Screening and difficulty prediction

**Part A — checks on the Day-1 items** (`day1_items.json` from Notebook 2):

1. *Sensibility* — is the correct answer correct, and does the question fit its category?
2. *Solvability* — is the item answerable from the text, and is the text needed at all?
3. *Distractor validity* — could a wrong option be defended as a second correct answer?
4. *Rule-based flaw checks* — six small functions, no model call.
5. *Model-based flaw checks* — five criteria from the item-writing-flaw literature (SAQUET),
   one short prompt each.

**Part B — difficulty prediction**: train a predictor on the public CMCQRD (Cambridge
Multiple-Choice Questions Reading Dataset: 120 reading tasks, 772 items at B1–C2 with a scaled
Rasch difficulty per item) from handcrafted text, option and TF-IDF n-gram features plus a
sentence embedding, check it with cross-validation, apply it to the Day-1 items.

*Notebook by Rasmus Jensen (ETH Zurich), adapted for the fall school by Rudolf Debelak (EPFL). The pipeline follows Jensen (2025, 2026).*


## Setup

Three things to fill in: the endpoint, the key, and the two model names (one model writes the
passages, one writes and judges the items). On Colab, store the key once under the key icon on
the left (*Secrets*) as `AIG_API_KEY` and the endpoint as `AIG_BASE_URL`; the cell reads them
from there. Everything else in the notebook only uses the three small helpers defined below:

* `render(template, **values)` fills a prompt template — `{{level}}` in the template becomes the
  value of `level`.
* `ask(system, user, schema)` sends one system message and one user message to the model. If a
  `schema` (a pydantic class) is given, the server is asked to reply in that JSON layout and the
  reply comes back as an object of that class, so the rest of the code never parses text.
  Every call is stored in `cache/`; an identical call is replayed from disk, so a notebook that
  has been run once runs again without the endpoint (set `OFFLINE = True` to enforce that).
  Failed calls are retried; a reply that does not fit the schema is asked for once more.
* `ask_many(calls)` runs a list of such calls in parallel, because a pipeline step usually asks
  the same question about many items.


In [ ]:
%pip install -q openai pydantic jinja2 pandas

import json, re, os, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
from jinja2 import Template
import pandas as pd

# --- endpoint, key, models -----------------------------------------------------------------
BASE_URL   = os.environ.get("AIG_BASE_URL", "https://YOUR-ENDPOINT/v1")   # the OpenAI-compatible endpoint
API_KEY    = os.environ.get("AIG_API_KEY", "EMPTY")
try:                                                     # on Colab: read both from the Secrets panel
    from google.colab import userdata
    API_KEY  = userdata.get("AIG_API_KEY") or API_KEY
    BASE_URL = userdata.get("AIG_BASE_URL") or BASE_URL
except Exception:
    pass
TEXT_MODEL = "swiss-ai/Apertus-70B-Instruct-2509"     # writes the passages
ITEM_MODEL = "Qwen/Qwen3-32B"                         # writes and judges the items (a thinking model)

# --- cache: every call is stored; identical calls are replayed ------------------------------
CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)
OFFLINE   = False                                     # True: never call the endpoint, only replay

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=240, max_retries=0)


def render(template, **values):
    return Template(template).render(**values)          # {{name}} in the template -> values[name]


def _parse(text, schema):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)   # thinking models: drop the reasoning block
    text = re.sub(r"```(?:json)?", "", text).strip()              # and code fences
    if schema is None:
        return text
    data = json.loads(text)
    if isinstance(data, list) and len(data) == 1:                # some models wrap the object in a list
        data = data[0]
    return schema.model_validate(data)


def ask(system, user, schema=None, model=None, temperature=0.4, seed=None, think=None, max_tokens=4096, retries=3):
    """One system + one user message -> plain text, or an instance of `schema`."""
    payload = dict(model=model or ITEM_MODEL, temperature=temperature, max_tokens=max_tokens,
                   messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    if schema is not None:                                        # ask the server for the schema's JSON layout
        payload["response_format"] = {"type": "json_schema",
                                      "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()}}
    if seed is not None:
        payload["seed"] = seed
    if think is not None:                                         # Qwen3-style thinking on/off (vLLM); ignored elsewhere
        payload["extra_body"] = {"chat_template_kwargs": {"enable_thinking": bool(think)}}

    path = CACHE_DIR / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:24] + ".json")
    for attempt in range(retries):
        cached = None
        if path.exists():
            try:
                cached = json.loads(path.read_text(encoding="utf-8"))["text"]
            except (OSError, ValueError):                         # half-written by a parallel call: treat as missing
                cached = None
        if cached is not None:
            text = cached
        elif OFFLINE:
            raise RuntimeError(f"OFFLINE and not in cache: {user[:80]!r}")
        else:
            try:
                reply = client.chat.completions.create(**payload)
                text = reply.choices[0].message.content or ""
            except Exception as e:                                # connection / server error: wait and retry
                time.sleep(3 * (attempt + 1))
                if attempt == retries - 1:
                    raise RuntimeError(f"endpoint failed {retries} times: {e}") from e
                continue
            tmp = path.with_name(path.name + f".{os.getpid()}.{id(payload)}.tmp")   # atomic write: parallel calls never see a half file
            tmp.write_text(json.dumps({"text": text, "payload": payload}, ensure_ascii=False), encoding="utf-8")
            os.replace(tmp, path)
        try:
            return _parse(text, schema)
        except (json.JSONDecodeError, ValidationError) as e:      # reply did not fit: drop it from the cache, ask again
            path.unlink(missing_ok=True)
            payload["temperature"] = 0.2
            if attempt == retries - 1:
                raise RuntimeError(f"reply did not match {schema.__name__}: {e}\n{text[:300]}") from e


def ask_many(calls, workers=6):                         # calls: list of dicts with ask()'s arguments
    with ThreadPoolExecutor(workers) as pool:
        return list(pool.map(lambda c: ask(**c), calls))


def need_file(name):
    """Make sure `name` exists in the session: on Colab, offer an upload dialog if it is missing."""
    if not Path(name).exists():
        try:
            from google.colab import files
            print(f"{name} not found - please upload it:")
            files.upload()
        except Exception:
            raise FileNotFoundError(f"{name} not found in the working directory")
    return name


In [ ]:
CATEGORIES = {
    'FR': {
        'name': 'Focus and Retrieve',
        'definition': 'The answer is stated in the text and the reader has to find it. Question and answer must be paraphrased, so that spotting a keyword is not enough.',
        'check': 'Does the question or the answer reuse the wording of the text? If yes, rephrase it.',
        'distractor': 'Each distractor must be something the text does NOT support anywhere.',
    },
    'INF': {
        'name': 'Inference',
        'definition': 'The answer is not stated in the text. The reader has to conclude it from clues the text gives.',
        'check': 'Could the answer be found by combining sentences of the text? If yes, it is not an inference - replace it.',
        'distractor': 'Each distractor must NOT be a valid conclusion from the text.',
    },
    'INT': {
        'name': 'Interpret and Integrate',
        'definition': 'The answer only follows from combining two different parts of the text. Neither part gives it alone.',
        'check': 'Could a reader who read only one of the two parts answer it? If yes, replace it.',
        'distractor': 'Each distractor is a wrong connection between the two parts, or a conclusion from one part only.',
    },
    'CRIT': {
        'name': 'Evaluate and Critique',
        'definition': 'The reader judges how the text is written: how effective an argument, an example, the structure or the ending is, and why.',
        'check': 'Is the answer a judgement about the writing rather than a summary of the content, and is the question not a yes/no question? If not, replace it.',
        'distractor': 'Each distractor must be a judgement the text clearly contradicts - not just a weaker judgement.',
    },
}

SENSIBLE_SYSTEM = """You are a strict reviewer of test questions."""

SENSIBLE_PROMPT = """Judge only the correct answer of this question: is it really correct according to the text, and does the question fit its category?

Category: {{name}}
{{definition}}

Score 1-10: 9-10 = clearly correct and fits the category; 7-8 = correct with a small flaw; 4-6 = doubtful; 1-3 = wrong, or the wrong category.

Text:
{{text}}

Question: {{question}}
Correct answer: {{answer}}"""

SOLVABLE_SYSTEM = """You judge whether questions can be answered from a text."""

SOLVABLE_PROMPT = """Give two scores from 1 to 10 for this question.
answerable_from_text: 10 = the text clearly gives the answer, 1 = the text does not help at all.
text_needed: 10 = nobody could answer this without the text, 1 = anyone could answer it from general knowledge.

Text:
{{text}}

Question: {{question}}
Answer: {{answer}}"""

VALIDITY_SYSTEM = """You are a strict reviewer of multiple-choice options."""

VALIDITY_PROMPT = """For each wrong option, rate from 1 to 10 how far a careful reader could defend it as ALSO correct.
1-2 = clearly wrong; 5-6 = partly defensible; 9-10 = it is a second correct answer.
An option that borrows facts from the text but does not answer the question is a good distractor (low score).

Text:
{{text}}

Question: {{question}}
Correct answer: {{answer}}
Wrong options:
{{options}}"""

FLAW_SYSTEM = """You review multiple-choice questions for writing flaws."""

FLAW_CRITERIA = {
    'clarity': 'Is each question clear and unambiguous? 10 = fully clear, 1 = confusing even for a good student.',
    'logical_cues': 'Does the wording of the question or the options give the answer away to a test-wise student, for example because the correct option is the longest or most detailed one? 10 = no cue, 1 = obvious cue.',
    'grammatical_cues': 'Do all options fit the question grammatically in the same way (tense, number, article)? 10 = fully parallel, 1 = the grammar reveals the answer.',
    'convergence_cues': "Do options overlap, contain each other or combine each other (like 'A and B')? 10 = all independent, 1 = strong overlap.",
    'implausible_distractors': 'Are the wrong options plausible? 10 = every wrong option could tempt a weak reader, 1 = at least one that nobody would choose.',
}

FLAW_PROMPT = """{{criterion}}
Give one score per question, in the order given.

{{items}}"""

## Load the Day-1 items

Upload `day1_items.json` (Notebook 2) to the session first. The reading items (`reading`) go
through all five checks; the math items from the template have no passage, so they only get the
rule-based and model-based flaw checks (checks 4 and 5). Two helpers: `passage_of(item)` returns
the passage text, `wrong_options(item)` the distractor texts.


In [ ]:
with open(need_file("day1_items.json"), encoding="utf-8") as f:
    day1 = json.load(f)

passages = {p["id"]: p for p in day1["passages"]}
items = day1["items"]                                                      # everything
reading = [it for it in items if it.get("passage") in passages]           # the reading items only
math = [it for it in items if it not in reading]


def passage_of(item):
    return passages[item["passage"]]["text"]


def wrong_options(item):
    return [t for L, t in item["options"].items() if L != item["correct"]]


print(len(reading), "reading items,", len(math), "math items")
pd.DataFrame(items)[["id", "cat", "question"]]

## Checks 1 and 2 · Sensibility and solvability

Both are one call per item. `sensibility_call(item)` fills the prompt with the category definition
and asks for a score 1–10 for the correct answer. `solvability_call(item)` asks for two scores:
`answerable_from_text` (does the text give the answer?) and `text_needed` (could someone answer
without the text?), which are averaged. An item passes at `PASS`.

In [ ]:
PASS = 7


class Score(BaseModel):
    reasoning: str
    score: int = Field(description="1 to 10.")

class Solvability(BaseModel):
    reasoning: str
    answerable_from_text: int = Field(description="1 to 10.")
    text_needed: int = Field(description="1 to 10.")


def sensibility_call(item):
    c = CATEGORIES[item["cat"]]
    prompt = render(SENSIBLE_PROMPT, name=c["name"], definition=c["definition"], text=passage_of(item),
                    question=item["question"], answer=item["options"][item["correct"]])
    return dict(system=SENSIBLE_SYSTEM, user=prompt, schema=Score, temperature=0.2)


def solvability_call(item):
    prompt = render(SOLVABLE_PROMPT, text=passage_of(item), question=item["question"], answer=item["options"][item["correct"]])
    return dict(system=SOLVABLE_SYSTEM, user=prompt, schema=Solvability, temperature=0.2)


results = {it["id"]: {} for it in items}
for it, r in zip(reading, ask_many([sensibility_call(it) for it in reading])):
    results[it["id"]]["sensibility"] = r.score
for it, r in zip(reading, ask_many([solvability_call(it) for it in reading])):
    results[it["id"]]["solvability"] = (r.answerable_from_text + r.text_needed) / 2
pd.DataFrame(results).T.loc[[it["id"] for it in reading]]

## Check 3 · Distractor validity

`validity_call(item)` shows the reviewer the three wrong options and gets one score per option: how
far could a careful reader defend it as *also* correct? The item's value is the maximum; it
fails when any option reaches `PASS` — that option is a second key.

In [ ]:
class OptionScores(BaseModel):
    scores: list[int] = Field(description="One score 1-10 per wrong option, in the order given.")


def validity_call(item):
    prompt = render(VALIDITY_PROMPT, text=passage_of(item), question=item["question"], answer=item["options"][item["correct"]],
                    options="\n".join(f"- {d}" for d in wrong_options(item)))
    return dict(system=VALIDITY_SYSTEM, user=prompt, schema=OptionScores, temperature=0.2)


calls = [validity_call(it) for it in reading]
for it, call, r in zip(reading, calls, ask_many(calls)):
    if len(r.scores) != len(wrong_options(it)):                  # wrong number of scores: ask once more, explicitly
        r = ask(call["system"], call["user"] + f"\n\nReturn exactly {len(wrong_options(it))} scores, one per wrong option.", schema=OptionScores, temperature=0.2)
    results[it["id"]]["max_defensible"] = max(r.scores[:len(wrong_options(it))])
pd.DataFrame(results).T.loc[[it["id"] for it in reading]]

## Check 4 · Rule-based flaw checks

Six functions, each returning `True` when the item is clean — the cheap half of the SAQUET
checks:

* `longest_is_key` — the correct option is much longer than every distractor (a classic giveaway);
* `all_or_none` — an option like "all of the above";
* `negative_stem` — the question contains *not / except / false*;
* `vague_words` — an option or the question uses *sometimes, usually, …*;
* `absolute_words_in_distractors` — *always, never, …* appear in distractors but not in the key;
* `word_repeat` — a content word of the question appears in the key and in no distractor.

In [ ]:
VAGUE = ["sometimes", "often", "usually", "rarely", "generally", "somewhat", "partly", "occasionally"]
ABSOLUTE = ["always", "never", "none", "all", "completely", "absolutely", "totally", "definitely"]
STOPWORDS = set("about after again also because before being between could does from have into more most other "
                "over some such than that their there these they this those through under what when where which while "
                "with would your text passage writer author according".split())


def content_words(s):
    return {w for w in re.findall(r"[a-z']+", s.lower()) if len(w) >= 4 and w not in STOPWORDS}


def longest_is_key(item):
    key = item["options"][item["correct"]]
    return not all(len(key) > 1.5 * len(d) for d in wrong_options(item))


def all_or_none(item):
    return not any("of the above" in t.lower() for t in item["options"].values())


def negative_stem(item):
    return not re.search(r"\b(not|except|false|incorrect)\b", item["question"].lower())


def vague_words(item):
    texts = [item["question"]] + list(item["options"].values())
    return not any(v in t.lower().split() for t in texts for v in VAGUE)


def absolute_words_in_distractors(item):
    key = set(item["options"][item["correct"]].lower().split())
    in_key = any(a in key for a in ABSOLUTE)
    in_distractors = any(a in d.lower().split() for d in wrong_options(item) for a in ABSOLUTE)
    return not (in_distractors and not in_key)


def word_repeat(item):
    shared = content_words(item["question"]) & content_words(item["options"][item["correct"]])
    shared = {w for w in shared if not any(w in content_words(d) for d in wrong_options(item))}
    return not shared


RULES = [longest_is_key, all_or_none, negative_stem, vague_words, absolute_words_in_distractors, word_repeat]
for it in items:
    results[it["id"]]["rule_fails"] = [f.__name__ for f in RULES if not f(it)]
pd.DataFrame(results).T[["rule_fails"]]

## Check 5 · Model-based flaw checks

Five criteria (`FLAW_CRITERIA`): clarity, logical cues, grammatical cues, convergence cues,
implausible distractors. `flaw_call(criterion, chunk)` shows up to twelve items to the model in one call and
gets one score per item back — cheaper than one call per item, and the model can compare.
The math items are included here.
An item fails a criterion below `PASS`.

In [ ]:
class ItemScores(BaseModel):
    scores: list[int] = Field(description="One score 1-10 per question, in the order given.")


def items_text(items):
    lines = []
    for i, it in enumerate(items, start=1):
        lines.append(f"Question {i}: {it['question']}\n   correct: {it['options'][it['correct']]}\n   wrong: {wrong_options(it)}")
    return "\n".join(lines)


def flaw_call(criterion, chunk):
    prompt = render(FLAW_PROMPT, criterion=criterion, items=items_text(chunk))
    return dict(system=FLAW_SYSTEM, user=prompt, schema=ItemScores, temperature=0.2)


CHUNK = 12
chunks = [items[i:i + CHUNK] for i in range(0, len(items), CHUNK)]
jobs = [(name, chunk) for name in FLAW_CRITERIA for chunk in chunks]
for (name, chunk), r in zip(jobs, ask_many([flaw_call(FLAW_CRITERIA[n], c) for n, c in jobs])):
    if len(r.scores) != len(chunk):                              # wrong number of scores: ask once more, explicitly
        call = flaw_call(FLAW_CRITERIA[name], chunk)
        r = ask(call["system"], call["user"] + f"\n\nReturn exactly {len(chunk)} scores, one per question.", schema=ItemScores, temperature=0.2)
    if len(r.scores) != len(chunk):
        raise ValueError(f"{name}: got {len(r.scores)} scores for {len(chunk)} items")
    for it, s in zip(chunk, r.scores):
        results[it["id"]][name] = s
pd.DataFrame(results).T

## Report

One row per item with every check and a verdict. In a real pipeline a failed item goes back to
generation; here it stays in the file with its flags, so that Notebook 4 can compare flagged and
clean items empirically.

In [ ]:
report = pd.DataFrame(results).T
for col in ["sensibility", "solvability", "max_defensible"] + list(FLAW_CRITERIA):
    report[col] = pd.to_numeric(report[col], errors="coerce")
is_reading = report.index.isin([it["id"] for it in reading])
reading_ok = (report["sensibility"] >= PASS) & (report["solvability"] >= PASS) & (report["max_defensible"] < PASS)
report["passed"] = ((reading_ok | ~is_reading)                                     # math items: no passage-based checks
                    & (report["rule_fails"].str.len() == 0)
                    & (report[list(FLAW_CRITERIA)].min(axis=1) >= PASS))
print(f"{int(report['passed'].sum())} of {len(report)} items pass every check they were given, at PASS = {PASS}")

for it in items:
    it["screening"] = {k: v for k, v in results[it["id"]].items()}
    it["screening"]["passed"] = bool(report.loc[it["id"], "passed"])
with open("day2_screened_items.json", "w", encoding="utf-8") as f:
    json.dump({"passages": day1["passages"], "items": items}, f, indent=2, ensure_ascii=False, default=str)
report

## Part B · The CMCQRD training data

The dataset (Mullooly et al., 2023, doi:10.17863/CAM.102185) is distributed by Cambridge
University Press & Assessment under a licence; download it and upload the file to the session.
Its documentation describes the file as JSON Lines, one *task* per line: a task has `id`, `title`,
the passage `text`, the target CEFR `level`, task-level `Diff` and `fac`, and a list of `questions`;
each question has its `text`, the `answer` letter (`a`–`d`), the item difficulty `Diff` (a Rasch
difficulty scaled so that most items fall between 0 and 100), `disc` and `fac`; each of the four
options has its `text` and, for 78 of the 120 tasks, its own `disc` and `fac` (`null` otherwise).
The loader below also accepts a JSON list of tasks and the three ways the options might be keyed
(a list under `options`, a dict keyed `a`–`d`, or `option_a` … `option_d`).
**Check the first record after loading** — the code raises a clear error if a key is missing.

`load_cmcqrd(path)` turns every question into one row of a table: task id, level, passage,
question, the text of the correct option, the three distractor texts, and the difficulty.


In [ ]:
CMCQRD_FILE = "CMCQRD.jsonl"        # or CMCQRD.json


def option_texts(question):                              # the four option texts in a-d order
    if "options" in question:
        opts = question["options"]
        if isinstance(opts, dict):                            # {"a": ..., "b": ...} or {"option_a": ..., ...}
            opts = [opts[k] for k in sorted(opts)]
    else:                                                    # "option_a" ... "option_d" next to the question fields
        opts = [question[k] for k in sorted(k for k in question if k.lower().startswith("option"))]
    if len(opts) != 4:
        raise ValueError(f"expected 4 options, found {len(opts)}: keys {list(question)}")
    return [o["text"] if isinstance(o, dict) else o for o in opts]


def key_index(answer):                                   # "a" / "A" / "option_a" / "(a)" -> 0
    letter = re.sub(r"[^a-d]", "", str(answer).lower().replace("option", ""))[:1]
    if not letter:
        raise ValueError(f"cannot read the answer letter from {answer!r}")
    return "abcd".index(letter)


def read_tasks(path):
    raw = Path(path).read_text(encoding="utf-8").strip()
    if raw.startswith("["):
        return json.loads(raw)
    return [json.loads(line) for line in raw.splitlines() if line.strip()]


def load_cmcqrd(path):
    rows = []
    for task in read_tasks(path):
        for q in task["questions"]:
            texts = option_texts(q)
            k = key_index(q["answer"])
            rows.append({"task": task["id"], "level": task["level"], "passage": task["text"],
                         "question": q["text"], "key": texts[k],
                         "distractors": [t for i, t in enumerate(texts) if i != k],
                         "difficulty": q.get("Diff", q.get("diff"))})
    df = pd.DataFrame(rows).dropna(subset=["difficulty"])
    if df.empty:
        raise ValueError("no items with a difficulty value - check the field names in the file")
    return df


tasks = read_tasks(need_file(CMCQRD_FILE))
print("first task keys:", list(tasks[0].keys()), "\nfirst question keys:", list(tasks[0]["questions"][0].keys()))
cm = load_cmcqrd(CMCQRD_FILE)
print(len(cm), "items from", cm["task"].nunique(), "tasks")
cm.groupby("level")["difficulty"].describe()[["count", "mean", "std"]]

## Features

Every item becomes one row of numbers. Six small functions build it:

* `text_features(text)` — counts and readability: words, average sentence length, average word
  length, share of long words, and four readability indices (Flesch reading ease, Gunning fog,
  SMOG, Dale–Chall — different formulas for the same idea, "how long are the words and the
  sentences, and how rare are the words").
* `pos_features(text)` — part-of-speech densities: the share of nouns, verbs, adjectives, adverbs
  and pronouns, from NLTK's tagger. A noun-heavy text is denser than a verb-heavy one.
* `sentiment_features(text)` — how positive, negative and subjective the text sounds (VADER and
  TextBlob). Not a difficulty measure in itself, but emotional or opinionated passages tend to
  come with harder items.
* `option_features(question, key, distractors)` — three numbers about the options: the key's
  length relative to the distractors (a much longer key is a giveaway, so the item gets easier),
  how many of the key's content words also appear in the question (word matching makes it
  easier), and how similar the distractors are to the key in meaning (close distractors make it
  harder).
* `TfidfNgrams(train_texts)` — words and word pairs weighted by TF-IDF: a word counts for more
  the more often it appears in this text and the rarer it is across all texts, so "the" and
  "and" weigh almost nothing and rare vocabulary weighs a lot — and rare vocabulary is what
  makes reading hard. The thousands of columns are compressed to 50 with a truncated SVD (LSA).
  Fitted on the training texts only, so nothing leaks from the test fold.
* `embed(df)` — a sentence embedding of `passage [SEP] question [SEP] key` from a small
  transformer; `trim()` first keeps the passage sentences most similar to the question so that
  the input fits.

`feature_table(df)` calls the first three for the passage and for question + key, adds the
option features, and returns one table.

In [ ]:
%pip install -q scikit-learn scipy textstat nltk textblob sentence-transformers
import numpy as np
import textstat, nltk
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sentence_transformers import SentenceTransformer, util

for package in ["punkt", "punkt_tab", "averaged_perceptron_tagger_eng", "vader_lexicon"]:
    nltk.download(package, quiet=True)
vader = SentimentIntensityAnalyzer()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def text_features(text):
    words = text.split()
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    return {"n_words": len(words),
            "sentence_length": len(words) / max(len(sentences), 1),
            "word_length": sum(len(w) for w in words) / max(len(words), 1),
            "long_words": sum(len(w) >= 7 for w in words) / max(len(words), 1),
            "flesch": textstat.flesch_reading_ease(text),
            "gunning_fog": textstat.gunning_fog(text),
            "smog": textstat.smog_index(text),
            "dale_chall": textstat.dale_chall_readability_score(text)}


def pos_features(text):
    tags = [tag for _, tag in nltk.pos_tag(nltk.word_tokenize(text))]
    n = max(len(tags), 1)
    return {"nouns": sum(t.startswith("NN") for t in tags) / n,
            "verbs": sum(t.startswith("VB") for t in tags) / n,
            "adjectives": sum(t.startswith("JJ") for t in tags) / n,
            "adverbs": sum(t.startswith("RB") for t in tags) / n,
            "pronouns": sum(t.startswith("PRP") for t in tags) / n}


def sentiment_features(text):
    v = vader.polarity_scores(text)
    blob = TextBlob(text).sentiment
    return {"negative": v["neg"], "positive": v["pos"], "compound": v["compound"],
            "polarity": blob.polarity, "subjectivity": blob.subjectivity}


def option_features(question, key, distractors):
    q_words = set(re.findall(r"[a-z]+", question.lower()))
    k_words = [w for w in re.findall(r"[a-z]+", key.lower()) if len(w) >= 4]
    sims = util.cos_sim(embedder.encode(key), embedder.encode(distractors))[0]     # meaning-closeness key vs distractors
    return {"key_length_ratio": len(key) / max(np.mean([len(d) for d in distractors]), 1),
            "key_question_overlap": sum(w in q_words for w in k_words) / max(len(k_words), 1),
            "distractor_similarity": float(sims.mean())}


def feature_table(df):
    rows = []
    for r in df.itertuples():
        row = {}
        for prefix, text in [("passage_", r.passage), ("item_", r.question + " " + r.key)]:
            for f in (text_features, pos_features, sentiment_features):
                row.update({prefix + k: v for k, v in f(text).items()})
        row.update(option_features(r.question, r.key, r.distractors))
        rows.append(row)
    return pd.DataFrame(rows)


class TfidfNgrams:
    def __init__(self, train_texts):
        self.tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True).fit(train_texts)  # words + word pairs
        self.svd = TruncatedSVD(50, random_state=0).fit(self.tfidf.transform(train_texts))                                # 5000 columns -> 50

    def transform(self, texts):
        return self.svd.transform(self.tfidf.transform(texts))


def trim(passage, question, budget=200):                # the sentences most similar to the question, up to `budget` words
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", passage) if s.strip()]
    sims = util.cos_sim(embedder.encode(question), embedder.encode(sentences))[0].tolist()
    keep, used = [], 0
    for i in sorted(range(len(sentences)), key=lambda i: -sims[i]):
        n = len(sentences[i].split())
        if used + n <= budget:
            keep.append(i); used += n
    return " ".join(sentences[i] for i in sorted(keep))


def embed(df):
    texts = [trim(p, q) + " [SEP] " + q + " [SEP] " + k for p, q, k in zip(df["passage"], df["question"], df["key"])]
    return embedder.encode(texts, batch_size=32, show_progress_bar=True)


X_hand = feature_table(cm)
X_emb = embed(cm)
print("handcrafted features:", X_hand.shape, " embeddings:", X_emb.shape)
X_hand.describe().T[["mean", "std", "min", "max"]].round(2)

## Train and evaluate

`evaluate(family)` runs a 5-fold cross-validation **grouped by task**: all items of a passage
stay in the same fold, otherwise the passage features would leak. For each fold it:
1. fits the TF-IDF n-grams on the training passages;
2. builds the feature matrix for the chosen family (`handcrafted` = the six functions above plus
   the level, `embedding`, or `all` = both stacked);
3. fits a support-vector regressor (SVR) on standardised features and predicts the test fold.

`level_mean` is the baseline you get for free: predict the mean difficulty of the item's CEFR
level. Metrics: Pearson correlation and RMSE on the CMCQRD scale.

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr

y = cm["difficulty"].astype(float).values
level_dummies = pd.get_dummies(cm["level"]).astype(float)


def features(family, idx, ngrams):
    hand = np.hstack([X_hand.values[idx], level_dummies.values[idx],
                      ngrams.transform(cm["passage"].iloc[idx] + " " + cm["question"].iloc[idx])])
    return {"handcrafted": hand, "embedding": X_emb[idx], "all": np.hstack([X_emb[idx], hand])}[family]


def evaluate(family):
    pred = np.zeros(len(y))
    for train, test in GroupKFold(5).split(cm, y, groups=cm["task"]):
        ngrams = TfidfNgrams((cm["passage"].iloc[train] + " " + cm["question"].iloc[train]).tolist())   # 1
        model = make_pipeline(StandardScaler(), SVR(C=10, epsilon=1))                                       # 3
        model.fit(features(family, train, ngrams), y[train])                                               # 2
        pred[test] = model.predict(features(family, test, ngrams))
    return pred


predictions = {"level_mean": np.zeros(len(y))}
for train, test in GroupKFold(5).split(cm, y, groups=cm["task"]):
    means = cm.iloc[train].groupby("level")["difficulty"].mean()
    predictions["level_mean"][test] = cm["level"].iloc[test].map(means).fillna(y[train].mean())
for family in ["handcrafted", "embedding", "all"]:
    predictions[family] = evaluate(family)

scores = pd.DataFrame({name: {"pearson": pearsonr(y, p)[0], "rmse": np.sqrt(np.mean((y - p) ** 2))}
                       for name, p in predictions.items()}).T.sort_values("pearson", ascending=False)
scores

## Apply to the Day-1 items

The best family is refitted on all CMCQRD items and applied to the Day-1 reading items (the math items have no passage), which are put
into the same table layout (`task` = passage id, `key` = the correct option's text,
`distractors` = the other three). The result is a difficulty on the CMCQRD scale per item,
saved to `day1_predictions.json` for Notebook 4.

In [ ]:
best = [f for f in scores.index if f != "level_mean"][0]
print("using", best)

ngrams = TfidfNgrams((cm["passage"] + " " + cm["question"]).tolist())
model = make_pipeline(StandardScaler(), SVR(C=10, epsilon=1)).fit(features(best, np.arange(len(y)), ngrams), y)

d1 = pd.DataFrame([{"task": it["passage"], "level": it["level"], "passage": passage_of(it), "question": it["question"],
                    "key": it["options"][it["correct"]], "distractors": wrong_options(it), "id": it["id"], "cat": it["cat"]}
                   for it in reading])
d1_hand = np.hstack([feature_table(d1).values, pd.get_dummies(d1["level"]).reindex(columns=level_dummies.columns, fill_value=0).values,
                     ngrams.transform(d1["passage"] + " " + d1["question"])])
d1_emb = embed(d1)
d1["predicted"] = model.predict({"handcrafted": d1_hand, "embedding": d1_emb, "all": np.hstack([d1_emb, d1_hand])}[best])

with open("day1_predictions.json", "w", encoding="utf-8") as f:
    json.dump({"model": best, "items": {r.id: float(r.predicted) for r in d1.itertuples()}}, f, indent=2)
print("saved day1_predictions.json")
d1[["id", "cat", "level", "predicted"]].sort_values("predicted")